In [13]:
!pip -q install folium plotly scikit-learn ipywidgets

In [ ]:
from google.colab import files
uploaded = files.upload()

# Select the uploaded CSV automatically
import pandas as pd
import numpy as np
import io

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Please upload crime_data.csv")

CSV_NAME = "crime_data.csv" if "crime_data.csv" in csv_files else csv_files[0]
df = pd.read_csv(io.BytesIO(uploaded[CSV_NAME]))

print("Loaded:", CSV_NAME)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
required_columns = ["Area", "Latitude", "Longitude", "Date", "Crime_Type"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df = df.dropna(subset=required_columns).copy()

# Keep valid geographic coordinates
df = df[
    df["Latitude"].between(-90, 90) &
    df["Longitude"].between(-180, 180)
].copy()

if df.empty:
    raise ValueError("No valid records remain after cleaning. Check the Date and coordinate columns.")

df["Hour"] = df["Date"].dt.hour
df["Day"] = df["Date"].dt.day_name()

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["Day"] = pd.Categorical(df["Day"], categories=day_order, ordered=True)

# Consistent category codes for the demonstration model
df["Crime_Code"], crime_categories = pd.factorize(df["Crime_Type"])
df["Area_Code"], area_categories = pd.factorize(df["Area"])

print(f"Clean records: {len(df):,}")
print(f"Areas: {df['Area'].nunique()} | Crime types: {df['Crime_Type'].nunique()}")
display(df[required_columns + ["Hour", "Day"]].head())

In [ ]:
import plotly.express as px
from IPython.display import display, Markdown

display(Markdown("## 📌 Dataset overview"))
total_crimes = len(df)
total_areas = df["Area"].nunique()
total_types = df["Crime_Type"].nunique()
peak_hour = int(df["Hour"].value_counts().idxmax())

display(pd.DataFrame({
    "Metric": ["Total crime records", "Areas represented", "Crime categories", "Busiest recorded hour"],
    "Value": [f"{total_crimes:,}", total_areas, total_types, f"{peak_hour:02d}:00–{peak_hour:59d}:59".replace(f"{peak_hour:59d}", f"{peak_hour:02d}")]
}))

In [ ]:
import folium
from folium.plugins import HeatMap

map_center = [df["Latitude"].mean(), df["Longitude"].mean()]
crime_map = folium.Map(location=map_center, zoom_start=11, tiles="OpenStreetMap")

heat_points = df[["Latitude", "Longitude"]].dropna().values.tolist()
HeatMap(heat_points, radius=12, blur=16, min_opacity=0.35).add_to(crime_map)

# Add a sample of individual incidents for context (keeps map responsive)
sample = df.sample(min(500, len(df)), random_state=42)
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=3,
        popup=f"Area: {row['Area']}<br>Crime: {row['Crime_Type']}<br>Date: {row['Date']}",
        color="red",
        fill=True,
        fill_opacity=0.55
    ).add_to(crime_map)

display(Markdown("## 🌍 Crime hotspot map"))
display(crime_map)

In [ ]:
display(Markdown("## 📊 Crime trend by day (%)"))
trend_df = (
    df.groupby(["Day", "Crime_Type"], observed=False)
      .size().reset_index(name="Count")
)
day_totals = trend_df.groupby("Day", observed=False)["Count"].transform("sum")
trend_df["Percentage"] = np.where(day_totals > 0, trend_df["Count"] / day_totals * 100, 0)
fig = px.bar(
    trend_df, x="Day", y="Percentage", color="Crime_Type",
    barmode="stack", category_orders={"Day": day_order},
    labels={"Percentage": "Share of recorded crimes (%)"},
    title="Crime composition for each weekday"
)
fig.update_layout(xaxis_title="Day", yaxis_title="Percentage (%)")
fig.show()

In [ ]:
display(Markdown("## 📊 Crime distribution by area (%)"))
area_df = df.groupby(["Area", "Crime_Type"]).size().reset_index(name="Count")
area_df["Percentage"] = area_df["Count"] / area_df.groupby("Area")["Count"].transform("sum") * 100

fig = px.bar(
    area_df, x="Area", y="Percentage", color="Crime_Type",
    barmode="stack", title="Crime category distribution within each area",
    labels={"Percentage": "Share of area's records (%)"}
)
fig.update_layout(xaxis_title="Area", yaxis_title="Percentage (%)", xaxis_tickangle=-40)
fig.show()

display(Markdown("## 📊 Overall crime category distribution"))
type_df = df["Crime_Type"].value_counts().rename_axis("Crime_Type").reset_index(name="Count")
fig = px.pie(type_df, names="Crime_Type", values="Count", hole=0.35, title="All recorded crimes by category")
fig.show()

In [ ]:
display(Markdown("## 🔍 Crime pattern analysis"))

hour_df = df.groupby("Hour").size().reindex(range(24), fill_value=0).reset_index(name="Count")
fig = px.line(hour_df, x="Hour", y="Count", markers=True, title="Recorded crimes by hour")
fig.update_xaxes(dtick=1)
fig.show()

day_pattern = df.groupby("Day", observed=False).size().reindex(day_order, fill_value=0).reset_index(name="Count")
fig = px.bar(day_pattern, x="Day", y="Count", title="Recorded crimes by weekday")
fig.show()

area_pattern = df.groupby("Area").size().sort_values(ascending=False).reset_index(name="Count")
fig = px.bar(area_pattern, x="Area", y="Count", title="Recorded crimes by area")
fig.update_layout(xaxis_tickangle=-40)
fig.show()

peak_rows = hour_df[hour_df["Count"] == hour_df["Count"].max()]
print("Busiest recorded hour(s):", ", ".join(f"{int(h):02d}:00" for h in peak_rows["Hour"]))

In [ ]:
display(Markdown("## 🛡️ Urban safety planning"))

safety_df = df.groupby(["Area", "Hour"]).size().reset_index(name="Crime_Count")
avg_count = safety_df["Crime_Count"].mean()

def classify_count(count):
    if count > 1.2 * avg_count:
        return "High"
    elif count < 0.8 * avg_count:
        return "Low"
    return "Medium"

safety_df["Risk_Level"] = safety_df["Crime_Count"].apply(classify_count)

for level in ["High", "Medium", "Low"]:
    print(f"{level} risk area-hour combinations: {(safety_df['Risk_Level'] == level).sum()}")

display(Markdown("### Highest recorded area/hour counts"))
display(safety_df.sort_values("Crime_Count", ascending=False).head(15).reset_index(drop=True))

display(Markdown("### Risk-level counts"))
risk_summary = safety_df["Risk_Level"].value_counts().reindex(["High", "Medium", "Low"], fill_value=0).rename_axis("Risk_Level").reset_index(name="Combinations")
display(risk_summary)

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

area_options = sorted(df["Area"].unique().tolist())
area_dropdown = widgets.Dropdown(options=area_options, description="Area:", layout=widgets.Layout(width="350px"))
hour_slider = widgets.IntSlider(value=20, min=0, max=23, step=1, description="Hour:", continuous_update=False)
check_button = widgets.Button(description="Check area", button_style="primary")
safety_output = widgets.Output()

def check_area(_):
    with safety_output:
        clear_output()
        selected_area = area_dropdown.value
        selected_hour = hour_slider.value
        match = safety_df[
            (safety_df["Area"] == selected_area) &
            (safety_df["Hour"] == selected_hour)
        ]
        if match.empty:
            print("No records for this area and hour in the dataset.")
            return
        row = match.iloc[0]
        print(f"Area: {selected_area}")
        print(f"Hour: {selected_hour:02d}:00–{selected_hour:02d}:59")
        print(f"Recorded incidents in dataset: {int(row['Crime_Count'])}")
        print(f"Relative category: {row['Risk_Level']} risk")
        print("This is a historical-data indicator, not a real-time safety assessment.")

check_button.on_click(check_area)
display(widgets.VBox([area_dropdown, hour_slider, check_button, safety_output]))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


model_df = df.copy()
model_df["Risk"] = 0
model_df.loc[(model_df["Hour"] >= 20) | (model_df["Hour"] <= 2), "Risk"] = 1
model_df.loc[model_df["Crime_Type"].isin(["Auto Theft", "Ganja Smuggling", "Robbery"]), "Risk"] = 1
model_df.loc[model_df["Area"].isin(["Chintal", "Kukatpally"]), "Risk"] = 1

X = model_df[["Area_Code", "Hour", "Crime_Code"]]
y = model_df["Risk"]

if y.nunique() < 2:
    print("The generated target contains only one class; the classifier cannot be trained.")
    model = None
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Demo holdout accuracy:", round(accuracy_score(y_test, y_pred), 3))
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Reminder: this score measures reproduction of the heuristic labels, not real-world crime forecasting.")

In [ ]:
area_pred = widgets.Dropdown(options=area_options, description="Area:", layout=widgets.Layout(width="350px"))
crime_options = sorted(df["Crime_Type"].unique().tolist())
crime_pred = widgets.Dropdown(options=crime_options, description="Crime:", layout=widgets.Layout(width="350px"))
hour_pred = widgets.IntSlider(value=22, min=0, max=23, description="Hour:", continuous_update=False)
predict_button = widgets.Button(description="Classify risk", button_style="warning")
predict_output = widgets.Output()

def predict_risk(_):
    with predict_output:
        clear_output()
        if model is None:
            print("Model is unavailable because the target has only one class.")
            return
        a_code = int(df.loc[df["Area"] == area_pred.value, "Area_Code"].iloc[0])
        c_code = int(df.loc[df["Crime_Type"] == crime_pred.value, "Crime_Code"].iloc[0])
        sample_input = pd.DataFrame([[a_code, hour_pred.value, c_code]],
                                    columns=["Area_Code", "Hour", "Crime_Code"])
        prediction = int(model.predict(sample_input)[0])
        probability = float(model.predict_proba(sample_input)[0][1])
        print("Model classification:", "Higher risk (class 1)" if prediction == 1 else "Lower risk (class 0)")
        print(f"Model probability for class 1: {probability:.1%}")
        print("This is a demonstration based on rule-generated training labels—not a verified prediction.")

predict_button.on_click(predict_risk)
display(widgets.VBox([area_pred, crime_pred, hour_pred, predict_button, predict_output]))

## Notes and limitations
- The notebook uses only the columns in your uploaded CSV.
- The heatmap visualizes the coordinates in the dataset; it is not a live crime feed.
- Safety categories are relative to the observed area/hour counts in this dataset.
- The Random Forest learns labels created by the original app's rules. For a genuine predictive model, you would need an independently defined target (for example, future incident counts by area and time) and time-based validation.
- Do not interpret a “Low” result as a guarantee of personal safety.